In [ ]:
import pandas as pd
import re

df_stats = pd.read_csv('/home/jovyan/sakhno/reasoning/data/gender/sakhno_stats_dataset_renamed.csv')

# LightGBM не поддерживает спецсимволы в названиях фичей — оставляем только буквы, цифры, пробел, _
def _lgb_safe_name(name):
    s = str(name)
    s = re.sub(r'[^\w\s]', '_', s, flags=re.UNICODE)  # не буква/цифра/подчёркивание/пробел → _
    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_+', '_', s).strip('_')
    s = s.replace(' ', '_')
    return s or 'unnamed'

df_stats.columns = [_lgb_safe_name(c) for c in df_stats.columns]
# устраняем дубликаты имён (на случай коллизий после замены)
seen = {}
new_cols = []
for c in df_stats.columns:
    if c in seen:
        seen[c] += 1
        new_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        new_cols.append(c)
df_stats.columns = new_cols

In [ ]:
df_stats = df_stats.drop(columns=[col for col in df_stats.columns if 'Std' in col])

In [ ]:
test_ids = pd.read_csv('/home/jovyan/sakhno/reasoning/data/gender/test_ids.csv')

In [ ]:
# Трейн по cl_id не из теста; медиана unique_transaction_days только по трейну; бинарный таргет
udd_col = 'Sum_of_expences_at_category__Cosmetics_stores'
test_id_values = test_ids.iloc[:, 0] if test_ids.shape[1] > 0 else test_ids['customer_id']
train_mask = ~df_stats['customer_id'].isin(test_id_values)
df_train = df_stats.loc[train_mask].copy()
df_test = df_stats.loc[~train_mask].copy()

median_udd_train = df_train[udd_col].median()
df_train['udd_above_median'] = (df_train[udd_col] > median_udd_train).astype(int)
# для теста тоже бинаризуем по той же границе (медиане трейна)
df_test['udd_above_median'] = (df_test[udd_col] > median_udd_train).astype(int)

print(f"Медиана {udd_col} на трейне: {median_udd_train}")
print(f"Трейн: {df_train['udd_above_median'].sum()} выше медианы из {len(df_train)}")
print(f"Тест:  {df_test['udd_above_median'].sum()} выше медианы из {len(df_test)}")

In [ ]:
# Whitebox-модель: предсказываем udd_above_median по остальным фичам (без unique_transaction_days)
# pip install autowoe  # если не установлен

from autowoe import AutoWoE
from sklearn.metrics import roc_auc_score
import numpy as np

exclude = {'customer_id', 'gender', udd_col, 'Mean_of_expences_at_category__Cosmetics_stores', 'Median_of_expences_at_category__Cosmetics_stores'}
whitebox_feature_cols = [c for c in df_train.columns
                         if c not in exclude
                         and pd.api.types.is_numeric_dtype(df_train[c])]
target_whitebox = 'udd_above_median'

# Заполняем пропуски для AutoWoE
X_train_wb = df_train[whitebox_feature_cols].copy()
X_train_wb = X_train_wb.replace([np.inf, -np.inf], [1e5, -1e5]).fillna(0)
X_test_wb = df_test[whitebox_feature_cols].copy()
X_test_wb = X_test_wb.replace([np.inf, -np.inf], [1e5, -1e5]).fillna(0)

features_type = {c: 'real' for c in whitebox_feature_cols if c != target_whitebox}

auto_woe = AutoWoE(
    task="BIN",
    interpreted_model=True,
    monotonic=False,
    max_bin_count=5,
    select_type=None,
    pearson_th=0.9,
    imp_th=0,
    th_const=32,
    force_single_split=True,
    th_nan=0.01,
    min_bin_size=0.01,
    oof_woe=True,
    n_folds=5,
    n_jobs=-1,
    regularized_refit=False,
    p_val=0.05,
    verbose=1,
)

auto_woe.fit(
    X_train_wb,
    target_name=target_whitebox,
    features_type=features_type,
)

In [ ]:
# Оценка whitebox-модели (предсказание бинарной фичи unique_transaction_days по остальным)
pred_train = auto_woe.predict_proba(X_train_wb)
pred_test = auto_woe.predict_proba(X_test_wb)

print("AUC ROC — трейн:", roc_auc_score(X_train_wb[target_whitebox], pred_train))
if len(df_test) > 0 and df_test[target_whitebox].nunique() > 1:
    print("AUC ROC — тест: ", roc_auc_score(df_test[target_whitebox], pred_test))

# Пример SQL-инференса (интерпретируемая модель)
print("\nПример SQL-запроса для инференса:")
print(auto_woe.get_sql_inference_query("table"))